# 01 · Define & Explore — target prep, PD-1-face hotspots, binder metrics

**Standard slot:** *define & explore.* **For Project 06 this means:** clean the PD-L1 ectodomain,
select the **hotspot residues on the PD-1-binding (competitive) face**, write down the binder metrics
+ cutoffs, and run a deterministic **mock** mini-run as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## The binder metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–target interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity. A passing design is a **hypothesis** until SPR/BLI.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target prep + PD-1-face hotspots

The design target is the **PD-L1 ectodomain (IgV domain)**, and the hotspots are the PD-L1 residues
that **PD-1 contacts** in the PD-1/PD-L1 complex — steering the binder there is what makes it a
*competitive blocker*. Fetch the candidate complexes with `data/download_data.py` (4ZQK / 5O45 —
**verify on RCSB**), isolate the PD-L1 chain, remove PD-1/waters/heteroatoms, and read the interface
residues off the complex.

Below we just *declare* an EXAMPLE hotspot set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual PD-1/PD-L1 interface** (numbering depends on the PDB you verify).

In [ ]:
import binder_tools as bt

TARGET = "PDL1"                      # cleaned PD-L1 IgV domain (you produce this from 4ZQK/5O45)
# EXAMPLE hotspots on the PD-1-binding face — VERIFY/REPLACE from the PD-1/PD-L1 interface (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
HOTSPOTS = bt.parse_hotspots("A56,A66,A115")   # EXAMPLE_DATA placeholder residues
print("target  :", TARGET)
print("hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified PD-1-face residues)")

## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the scorer). The **mock** backend is deterministic and GPU-free so you can develop the plumbing.
**Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [ ]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

## 3 · Epitope-competition proxy (does it cover the PD-1 footprint?)

A binder only *blocks* PD-1 if it overlaps the PD-1 footprint enough. `hotspot_overlap()` is a
geometry proxy (fraction of hotspots contacted) — a teaching stand-in for the PD-1-competition assay
in notebook 04. Higher ⇒ more likely to block (not a guarantee).

In [ ]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> PD-1-footprint overlap = {ov} (SYNTHETIC)")

## Visualize a binder–target complex (py3Dmol)

Use this to eyeball a predicted binder–PD-L1 complex once you have a real PDB (from AF2-Multimer).

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] PD-1/PD-L1 accessions verified on RCSB (4ZQK/5O45 are candidates); PD-L1 chain + IgV domain identified.
- [ ] Cleaned PD-L1 target + **PD-1-face hotspot list** (derived from the interface, not invented).
- [ ] One-paragraph definition of each binder metric **with** its "does not mean" note.
- [ ] Reproduced mock mini-run (both paradigms) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign.